In [1]:
import os, subprocess, json, time, random, glob
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

import torch
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

subprocess.run("apt-get install -y -qq aria2 > /dev/null 2>&1", shell=True)
print("aria2 path:", subprocess.run(["which", "aria2c"], capture_output=True, text=True).stdout.strip())

Mounted at /content/drive
GPU available: True | Tesla T4
aria2 path: /usr/bin/aria2c


In [2]:
import requests
data = requests.get("https://zenodo.org/api/records/6906306").json()
la_file = next(f for f in data["files"] if f["key"] == "LA.zip")
download_url = la_file["links"]["self"]
expected_bytes = la_file["size"]
expected_md5 = (la_file.get("checksum") or "").replace("md5:", "")

print("File:", la_file["key"], "| size:", round(expected_bytes / 1024**3, 2), "GB")
print("URL:", download_url)
print("md5:", expected_md5 or "(not provided)")

File: LA.zip | size: 7.12 GB
URL: https://zenodo.org/api/records/6906306/files/LA.zip/content
md5: 30c98f11d8b2bc21f2c257bfd78bb5c5


In [33]:
os.makedirs("/content/asvspoof", exist_ok=True)
zip_path = "/content/asvspoof/LA.zip"
log_path = "/content/asvspoof/download.log"

try:
    already = proc.poll() is None
except NameError:
    already = False

if already:
    print("A download is already running. Do NOT start another. Go to Cell 3.")
else:
    cmd = ["aria2c", "-x", "4", "-s", "4", "-k", "20M", "-c", "--file-allocation=none",
           "--max-tries=20", "--retry-wait=5", "--console-log-level=warn",
           "-d", "/content/asvspoof", "-o", "LA.zip", download_url]
    proc = subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    print("Download started in the background. pid =", proc.pid)

A download is already running. Do NOT start another. Go to Cell 3.


In [53]:
def written_mb():
    if not os.path.exists(zip_path): return 0.0
    return os.stat(zip_path).st_blocks * 512 / 1024**2   # real bytes on disk

total_mb = expected_bytes / 1024**2
a = written_mb(); time.sleep(20); b = written_mb()
speed = (b - a) / 20
print(f"{b:,.0f} / {total_mb:,.0f} MB  ({b / total_mb * 100:.1f}%)")
print(f"speed: {speed:.1f} MB/s", end="")
print(f" | about {(total_mb - b) / speed / 60:.0f} min left" if speed > 0.05 else " | not moving")
print("download process:", "running" if proc.poll() is None else f"finished (exit code {proc.returncode})")

7,287 / 7,287 MB  (100.0%)
speed: 0.0 MB/s | not moving
download process: finished (exit code 0)


In [54]:
import signal
for pid in subprocess.run(["pgrep", "-x", "wget"], capture_output=True, text=True).stdout.split():
    os.kill(int(pid), signal.SIGKILL)
time.sleep(2)
print(subprocess.run(["pgrep", "-a", "wget"], capture_output=True, text=True).stdout or "no wget running")

no wget running


In [55]:
if proc.poll() is None:
    print("Still downloading. Wait and re-run Cell 3.")
else:
    assert proc.returncode == 0, "Download failed. Check the log."
    actual = os.path.getsize(zip_path)
    print("Size matches Zenodo:", actual == expected_bytes, f"({actual:,} vs {expected_bytes:,})")
    if expected_md5:
        import hashlib
        h = hashlib.md5()
        with open(zip_path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 24), b""):
                h.update(chunk)
        print("md5 matches Zenodo:", h.hexdigest() == expected_md5)

Size matches Zenodo: True (7,640,952,520 vs 7,640,952,520)
md5 matches Zenodo: True


In [56]:
assert os.path.getsize(zip_path) == expected_bytes, "Zip is incomplete. Go back to Cell 4b."

def find_base(root="/content/asvspoof"):
    for dp, dn, fn in os.walk(root):
        if "ASVspoof2019_LA_train" in dn:
            return dp
    return None

base = find_base()
if base is None:
    subprocess.run(["unzip", "-q", "-o", zip_path, "-d", "/content/asvspoof/"], check=True)
    base = find_base()
print("Base folder:", base)
print(sorted(os.listdir(base)))

Base folder: /content/asvspoof/LA
['ASVspoof2019_LA_asv_protocols', 'ASVspoof2019_LA_asv_scores', 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019_LA_dev', 'ASVspoof2019_LA_eval', 'ASVspoof2019_LA_train', 'README.LA.txt']


In [57]:
def parse_protocol(path):
    items = {}
    with open(path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 2:
                continue
            items[p[1]] = {"label": 0 if p[-1] == "bonafide" else 1,
                           "attack": p[3] if len(p) >= 5 else "-"}
    return items

def find_file(pattern):
    return glob.glob(os.path.join(base, "**", pattern), recursive=True)[0]

train_items = parse_protocol(find_file("*cm.train.trn.txt"))
dev_items   = parse_protocol(find_file("*cm.dev.trl.txt"))
eval_items  = parse_protocol(find_file("*cm.eval.trl.txt"))

for name, it in [("Train", train_items), ("Dev", dev_items), ("Eval", eval_items)]:
    nb = sum(v["label"] == 0 for v in it.values())
    atks = sorted({v["attack"] for v in it.values() if v["label"] == 1})
    print(f"{name}: {len(it)} | real {nb} | fake {len(it) - nb} | attacks: {atks}")

Train: 25380 | real 2580 | fake 22800 | attacks: ['A01', 'A02', 'A03', 'A04', 'A05', 'A06']
Dev: 24844 | real 2548 | fake 22296 | attacks: ['A01', 'A02', 'A03', 'A04', 'A05', 'A06']
Eval: 71237 | real 7355 | fake 63882 | attacks: ['A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19']


In [58]:
def subsample(items, n_bonafide, n_spoof, seed=42):
    r = random.Random(seed)
    bona = sorted(k for k, v in items.items() if v["label"] == 0)
    byatk = {}
    for k, v in items.items():
        if v["label"] == 1:
            byatk.setdefault(v["attack"], []).append(k)
    r.shuffle(bona)
    chosen = bona[:n_bonafide]
    per = max(n_spoof // max(len(byatk), 1), 1)
    for a in sorted(byatk):
        lst = sorted(byatk[a]); r.shuffle(lst)
        chosen += lst[:per]
    return {k: items[k] for k in chosen}

train_sub = subsample(train_items, 2580, 10200)
dev_sub   = subsample(dev_items,   2548, 4800)
test_sub  = subsample(eval_items,  800,  3200)

train_dir = os.path.join(base, "ASVspoof2019_LA_train", "flac")
dev_dir   = os.path.join(base, "ASVspoof2019_LA_dev", "flac")
eval_dir  = os.path.join(base, "ASVspoof2019_LA_eval", "flac")

for name, s in [("Train", train_sub), ("Dev", dev_sub), ("Test", test_sub)]:
    nb = sum(v["label"] == 0 for v in s.values())
    print(f"{name}: {len(s)} | real {nb} | fake {len(s) - nb}")

Train: 12780 | real 2580 | fake 10200
Dev: 7348 | real 2548 | fake 4800
Test: 3998 | real 800 | fake 3198


In [59]:
import librosa
from joblib import Parallel, delayed

N_MFCC, MAX_FRAMES, HOP = 40, 400, 160

def extract_mfcc(path):
    try:
        y, sr = librosa.load(path, sr=16000, duration=8.0)
        y, _ = librosa.effects.trim(y, top_db=30)
        if len(y) < 400:
            return None
        m = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=512, hop_length=HOP)
        if m.shape[1] < MAX_FRAMES:
            m = np.pad(m, ((0, 0), (0, MAX_FRAMES - m.shape[1])), mode="constant")
        else:
            m = m[:, :MAX_FRAMES]
        return m.astype(np.float32)
    except Exception:
        return None

for n in list(train_sub)[:3]:
    f = extract_mfcc(os.path.join(train_dir, n + ".flac"))
    print(n, "->", None if f is None else f.shape)

LA_T_5439881 -> (40, 400)
LA_T_2084171 -> (40, 400)
LA_T_1158098 -> (40, 400)


In [60]:
def build(items, flac_dir):
    names = list(items.keys())
    feats = Parallel(n_jobs=-1)(delayed(extract_mfcc)(os.path.join(flac_dir, n + ".flac")) for n in names)
    keep = [i for i, f in enumerate(feats) if f is not None]
    X = np.stack([feats[i] for i in keep])
    y = np.array([items[names[i]]["label"] for i in keep])
    atk = np.array([items[names[i]]["attack"] for i in keep])
    nm = np.array([names[i] for i in keep])
    print(f"  kept {len(keep)} of {len(names)} files | shape {X.shape}")
    return X, y, nm, atk

t0 = time.time()
print("TRAIN..."); X_train, y_train, names_train, atk_train = build(train_sub, train_dir)
print("DEV...");   X_val,   y_val,   names_val,   atk_val   = build(dev_sub, dev_dir)
print("TEST...");  X_test,  y_test,  names_test,  atk_test  = build(test_sub, eval_dir)
print(f"Done in {(time.time() - t0) / 60:.1f} min")

np.savez_compressed("/content/drive/MyDrive/voice_features.npz",
    X_train=X_train.astype(np.float16), y_train=y_train, names_train=names_train, atk_train=atk_train,
    X_val=X_val.astype(np.float16),     y_val=y_val,     names_val=names_val,     atk_val=atk_val,
    X_test=X_test.astype(np.float16),   y_test=y_test,   names_test=names_test,   atk_test=atk_test)
print("Saved voice_features.npz to Drive")

TRAIN...
  kept 12780 of 12780 files | shape (12780, 40, 400)
DEV...
  kept 7348 of 7348 files | shape (7348, 40, 400)
TEST...
  kept 3998 of 3998 files | shape (3998, 40, 400)
Done in 2.5 min
Saved voice_features.npz to Drive


In [61]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import copy, pandas as pd
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix, roc_curve

d = np.load("/content/drive/MyDrive/voice_features.npz")
X_train, y_train = d["X_train"].astype(np.float32), d["y_train"]
X_val,   y_val   = d["X_val"].astype(np.float32),   d["y_val"]
X_test,  y_test  = d["X_test"].astype(np.float32),  d["y_test"]
names_val, names_test, atk_test = d["names_val"], d["names_test"], d["atk_test"]

mu = X_train.mean(axis=(0, 2), keepdims=True)
sd = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
def prep(X): return torch.tensor((X - mu) / sd).unsqueeze(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = DataLoader(TensorDataset(prep(X_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(prep(X_val),   torch.tensor(y_val)),   batch_size=128)
test_loader  = DataLoader(TensorDataset(prep(X_test),  torch.tensor(y_test)),  batch_size=128)

n_real, n_fake = (y_train == 0).sum(), (y_train == 1).sum()
class_weights = torch.tensor([len(y_train) / (2 * n_real), len(y_train) / (2 * n_fake)], dtype=torch.float32).to(device)
print("Class weights (real, fake):", [round(w, 3) for w in class_weights.tolist()])

class VoiceCNN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.conv = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), nn.AdaptiveAvgPool2d((4, 4)))
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(64 * 4 * 4, 2))
    def forward(self, x):
        return self.fc(self.conv(x))

torch.manual_seed(42)
model = VoiceCNN().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print("Train/Dev/Test sizes:", len(y_train), len(y_val), len(y_test), "| device:", device)

Class weights (real, fake): [2.477, 0.626]
Train/Dev/Test sizes: 12780 7348 3998 | device: cuda


In [62]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        out = model(x.to(device))
        probs.extend(torch.softmax(out, dim=1)[:, 1].cpu().tolist())
        labels.extend(y.tolist())
    return np.array(probs), np.array(labels)

def compute_eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return (fpr[i] + fnr[i]) / 2

num_epochs = 15
best_eer, best_epoch, best_state = 1.0, 0, None

for epoch in range(num_epochs):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item()

    probs, lab = predict(model, val_loader)
    eer = compute_eer(lab, probs)
    bal = balanced_accuracy_score(lab, (probs >= 0.5).astype(int))
    print(f"Epoch {epoch+1}/{num_epochs} | train loss {running/len(train_loader):.4f} | dev EER {eer*100:.2f}% | dev bal-acc {bal*100:.2f}%")

    if eer < best_eer:
        best_eer, best_epoch = eer, epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, "/content/drive/MyDrive/voice_model.pth")

model.load_state_dict(best_state)
print(f"\nBest epoch: {best_epoch} (dev EER {best_eer*100:.2f}%)")

Epoch 1/15 | train loss 0.5275 | dev EER 29.72% | dev bal-acc 50.00%
Epoch 2/15 | train loss 0.3241 | dev EER 11.50% | dev bal-acc 84.20%
Epoch 3/15 | train loss 0.2288 | dev EER 9.31% | dev bal-acc 89.50%
Epoch 4/15 | train loss 0.1813 | dev EER 20.29% | dev bal-acc 51.12%
Epoch 5/15 | train loss 0.1560 | dev EER 10.97% | dev bal-acc 75.54%
Epoch 6/15 | train loss 0.1385 | dev EER 10.33% | dev bal-acc 82.35%
Epoch 7/15 | train loss 0.1178 | dev EER 11.25% | dev bal-acc 66.05%
Epoch 8/15 | train loss 0.1196 | dev EER 11.38% | dev bal-acc 72.98%
Epoch 9/15 | train loss 0.1071 | dev EER 9.34% | dev bal-acc 71.92%
Epoch 10/15 | train loss 0.0981 | dev EER 9.43% | dev bal-acc 71.95%
Epoch 11/15 | train loss 0.0882 | dev EER 6.11% | dev bal-acc 92.67%
Epoch 12/15 | train loss 0.0819 | dev EER 11.30% | dev bal-acc 61.75%
Epoch 13/15 | train loss 0.0971 | dev EER 8.09% | dev bal-acc 91.32%
Epoch 14/15 | train loss 0.0762 | dev EER 11.17% | dev bal-acc 87.53%
Epoch 15/15 | train loss 0.0778 | 

In [63]:
def summarize(probs, labels, title):
    preds = (probs >= 0.5).astype(int)
    print(title)
    print(f"  EER: {compute_eer(labels, probs)*100:.2f}%   (lower is better, 50% = guessing)")
    print(f"  Balanced accuracy: {balanced_accuracy_score(labels, preds)*100:.2f}%")
    print(f"  Real correct: {(preds[labels==0]==0).mean()*100:.2f}% | Fake correct: {(preds[labels==1]==1).mean()*100:.2f}%")
    print(f"  AUC: {roc_auc_score(labels, probs):.4f}")
    print(f"  Confusion matrix:\n{confusion_matrix(labels, preds)}")

val_probs, val_lab = predict(model, val_loader)
summarize(val_probs, val_lab, "DEV (same attack types as training)")

test_probs, test_lab = predict(model, test_loader)
summarize(test_probs, test_lab, "TEST (eval partition, mostly UNSEEN attack types)")

test_df = pd.DataFrame({"file": names_test, "attack": atk_test, "label": test_lab, "spoof_prob": test_probs})
print("\nFakes detected per attack type (threshold 0.5):")
print(test_df[test_df.label == 1].groupby("attack")["spoof_prob"].agg(n="count", detected=lambda s: (s >= 0.5).mean() * 100).round(1))

DEV (same attack types as training)
  EER: 6.11%   (lower is better, 50% = guessing)
  Balanced accuracy: 92.67%
  Real correct: 89.64% | Fake correct: 95.71%
  AUC: 0.9848
  Confusion matrix:
[[2284  264]
 [ 206 4594]]
TEST (eval partition, mostly UNSEEN attack types)
  EER: 21.44%   (lower is better, 50% = guessing)
  Balanced accuracy: 78.96%
  Real correct: 90.38% | Fake correct: 67.54%
  AUC: 0.8586
  Confusion matrix:
[[ 723   77]
 [1038 2160]]

Fakes detected per attack type (threshold 0.5):
          n  detected
attack               
A07     246      87.4
A08     246     100.0
A09     246      95.9
A10     246      28.9
A11     246      77.6
A12     246       8.1
A13     246      26.8
A14     246      68.7
A15     246      44.7
A16     246      91.9
A17     246      79.3
A18     246      73.2
A19     246      95.5


In [64]:
val_df = pd.DataFrame({"file": names_val, "label": val_lab, "spoof_prob": val_probs})
val_df.to_csv("/content/drive/MyDrive/voice_val_scores.csv", index=False)
test_df.to_csv("/content/drive/MyDrive/voice_test_scores.csv", index=False)

info = {
    "model": "VoiceCNN (3 conv blocks + BatchNorm) on MFCC, 40 coeffs x 400 frames (4 s), silence trimmed",
    "dataset": "ASVspoof2019 LA, official train/dev/eval; attack-stratified subsets",
    "subset_sizes": {"train": int(len(y_train)), "dev": int(len(y_val)), "test": int(len(y_test))},
    "note": "Test uses eval partition: mostly attack types NOT seen in training",
    "norm_mean": mu.flatten().tolist(), "norm_std": sd.flatten().tolist(),
    "mfcc": {"sr": 16000, "n_mfcc": N_MFCC, "n_fft": 512, "hop_length": HOP, "max_frames": MAX_FRAMES, "trim_top_db": 30},
    "label_map": {"real": 0, "fake": 1},
    "class_weights": [class_weights[0].item(), class_weights[1].item()],
    "learning_rate": 0.001, "batch_size": 64, "seed": 42,
    "epochs_run": num_epochs, "best_epoch": best_epoch,
    "val_eer": float(compute_eer(val_lab, val_probs)),
    "val_auc": float(roc_auc_score(val_lab, val_probs)),
    "test_eer": float(compute_eer(test_lab, test_probs)),
    "test_auc": float(roc_auc_score(test_lab, test_probs)),
    "test_balanced_acc": float(balanced_accuracy_score(test_lab, (test_probs >= 0.5).astype(int))),
}
with open("/content/drive/MyDrive/voice_model_info.json", "w") as f:
    json.dump(info, f, indent=2)
print("Saved. Test EER:", round(info["test_eer"] * 100, 2), "% | Test AUC:", round(info["test_auc"], 4))

Saved. Test EER: 21.44 % | Test AUC: 0.8586
